# AskSight — model training (Colab T4)

``
photo → CLIP (frozen) → connector (trained) → Qwen2.5-1.5B (frozen) → answer text
``

Run all cells with defaults in **≈ 30–40 min**. Deliverables: the checkpoint zip (`checkpoints/<VERSION>/`, a few MB) and held-out **VQA accuracy + latency** in `metrics.json`.

Scope: short factual questions about photos (VQA v2, English) — not open-ended description.

## 0. Configuration

- `VERSION` — checkpoint folder name (`checkpoints/<VERSION>/`); bump it to avoid overwriting a good checkpoint.
- `TRAIN / VAL / TEST` — question counts; defaults 2000/250/250. Fuller run: 20000/1000/1000 (≈ 3–6 h).
- `DTYPE` — `float16` (free T4s have no native bf16; `auto` would pick slow float32). Use `float32` if the loss becomes NaN.
- `SEED` — `0` fixes the train/val/test split (same fingerprint as local runs).

In [ ]:
VERSION = "v0.1-colab"
TRAIN, VAL, TEST = 2000, 250, 250
DTYPE = "float16"
SEED = 0

## 1. Install (~2 min)

Clones the repo and installs the `asksight_model` package (torch, transformers, pillow). Re-run safe; no large downloads on Colab.

In [ ]:
import os
if not os.path.isdir("AskSight"):
    !git clone https://github.com/Pranjal6955/AskSight.git
%cd AskSight/model
!pip install -q -e ".[dev]"

## 2. Verify GPU

Asserts CUDA is available; prints the assigned GPU (expect `Tesla T4`). If it fails: **Runtime → Change runtime type → T4 GPU**, then re-run.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable GPU first: Runtime → Change runtime type → T4 GPU"

## 3. Dataset (~5–10 min)

Downloads official VQA v2 questions/annotations plus only the COCO images the subset uses. `SEED = 0` makes the split deterministic and image-disjoint (no train→test image leakage); sha256 fingerprints land in `meta.json`.

In [ ]:
%%time
!python -m asksight_model.data --out data/vqa --train {TRAIN} --val {VAL} --test {TEST} --seed {SEED}
!cat data/vqa/meta.json

## 4. Train (~15–25 min)

2 epochs, batch 4, AdamW with gradient clipping and per-epoch validation — only the connector learns. Output: `checkpoints/<VERSION>/` (`projector.safetensors`, `config.yaml`, `metrics.json`, `data_meta.json`).

Watch the loss trend down. If it becomes `nan`, set `DTYPE = "float32"` in cell 0 and re-run. On CUDA OOM: Runtime → Restart, then re-run from step 1.

In [ ]:
%%time
!python -m asksight_model.train --config configs/default.yaml --dtype {DTYPE} --version {VERSION}

## 5. Evaluate (~2 min)

Official VQA accuracy (`min(matching/3, 1)` against 10 human references per question) on the held-out test split, plus median/p95 latency — recorded into `metrics.json` under `eval_test`.

In [ ]:
%%time
!python -m asksight_model.evaluate --checkpoint checkpoints/{VERSION} --split test --data-dir data/vqa
import json
ev = json.load(open(f"checkpoints/{VERSION}/metrics.json"))["eval_test"]
print(f"\nVQA accuracy: {ev['vqa_accuracy']:.1%} | latency median {ev['latency_median_s']}s, p95 {ev['latency_p95_s']}s")

## 6. Download the checkpoint

Zips `checkpoints/<VERSION>/` and downloads it. Colab storage is ephemeral — the zip is the deliverable. It contains the trained connector weights, configs, metrics, and dataset provenance.

In [ ]:
!zip -qr {VERSION}-ckpt.zip checkpoints/{VERSION}
import os
if os.environ.get("COLAB_RELEASE_TAG"):  # real Colab runtime
    from google.colab import files
    files.download(f"{VERSION}-ckpt.zip")
else:
    size = os.path.getsize(f"{VERSION}-ckpt.zip") / 1e6
    print(f"ready: {os.path.abspath(f'{VERSION}-ckpt.zip')} ({size:.1f} MB)")

## 7. Serve the checkpoint (laptop / server)

```bash
cd ~/AskSight-checkpoints && unzip ~/Downloads/v0.1-colab-ckpt.zip
cd backend && source venv/bin/activate
pip install -e ../model
export ASKSIGHT_MODEL_CHECKPOINT=$HOME/AskSight-checkpoints/checkpoints/v0.1-colab
uvicorn app.main:app --reload   # /ask now returns real answers
```

Next training run only if quality demands it: `TRAIN = 20000`, or more epochs if the final `val_loss` was still falling.